# 作业练习

把第 1 天的“网页摘要”项目升级为：用通过 Ollama 在本地运行的开源模型，而不是 OpenAI。

后续项目如果想避免付费 API，也可以沿用同样做法。

**优点：**
1. 无 API 费用——开源
2. 数据不离开你的电脑

**缺点：**
1. 能力明显弱于前沿闭源模型

## Ollama 安装回顾

直接访问 [ollama.com](https://ollama.com) 安装即可！

安装完成后，ollama 服务通常已在本地运行。  
若访问：  
[http://localhost:11434/](http://localhost:11434/)

应看到 `Ollama is running`。  

若没有，打开新的 Terminal（Mac）或 Powershell（Windows）并输入 `ollama serve`  
再在另一个终端输入 `ollama pull llama3.2`  
然后再次访问 [http://localhost:11434/](http://localhost:11434/)。

如果本机上 Ollama 很慢，可改用 `llama3.2:1b`：在终端执行 `ollama pull llama3.2:1b`，并把代码里的 `MODEL = "llama3.2"` 改成 `MODEL = "llama3.2:1b"`。



In [ ]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
# 通过 OpenAI 兼容接口连接本地 Ollama
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"  # 必填但可为占位值
)



In [ ]:
# 检查 Ollama 是否在运行
requests.get("http://localhost:11434").content



In [ ]:
!ollama pull llama3.2:1b

In [ ]:
def fetch_website_contents(url):
    # 抓取网页文本并截断，加快后续推理
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    text = soup.get_text(separator="\n")
    
    return text[:5000]  # 为提速做截断



In [ ]:
def build_messages(website_text):
    # 构造系统/用户消息
    return [
        {
            "role": "system",
            "content": "You are a helpful assistant that summarizes websites clearly and concisely."
        },
        {
            "role": "user",
            "content": f"Summarize this website:\n\n{website_text}"
        }
    ]



In [ ]:
def summarize_with_ollama(url):
    # 抓取 → 组消息 → 本地模型摘要
    website = fetch_website_contents(url)
    messages = build_messages(website)

    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=messages
    )

    return response.choices[0].message.content



In [ ]:
def display_summary(url):
    # 以 Markdown 展示摘要
    summary = summarize_with_ollama(url)
    display(Markdown(summary))



In [ ]:
display_summary("https://edwarddonner.com")